# AG_PRAXIS NB11 — Pilot: Loading Windows With Their Capture Identity

Each attack class in this dataset was recorded in its own session, and eight of the
nineteen classes were recorded more than once. That matters because it is the only place
in the corpus where which session a window came from and which class it belongs to are
not the same fact. On a class recorded once, knowing the session tells you the class and
knowing the class tells you the session, so nothing can be measured about one while
holding the other still. On a class recorded several times there is a real question:
given a window of this class, which of its sessions did it come from?

This notebook is the first step of an experiment that needs that structure. Before
anything is trained I want to see the ground it stands on, so all this does is read the
saved windows together with the recording each one was cut from, work the recording names
back to session identifiers, and count what is actually there.

Nothing is trained here and nothing is written to disk. It prints the shapes of the
arrays, how many sessions the training partition holds, and how many training windows
belong to the eight classes recorded more than once.

The usual first cell: Drive, the repository, and the commit this ran at.

In [3]:
import os
import subprocess
import sys
from datetime import date
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/AGREWAL14/AG_PRAXIS.git"

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    REPO_ROOT = Path("/content/repo")
    if REPO_ROOT.exists():
        subprocess.run(["git", "-C", str(REPO_ROOT), "pull", "--ff-only"], check=True)
    else:
        subprocess.run(["git", "clone", REPO_URL, str(REPO_ROOT)], check=True)
else:
    REPO_ROOT = Path.cwd()
    while not (REPO_ROOT / "config" / "base.yaml").exists() and REPO_ROOT != REPO_ROOT.parent:
        REPO_ROOT = REPO_ROOT.parent

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))


def git(*args):
    return subprocess.run(
        ["git", "-C", str(REPO_ROOT), *args], capture_output=True, text=True
    ).stdout.strip()


GIT_SHA = git("rev-parse", "--short", "HEAD")
GIT_BRANCH = git("rev-parse", "--abbrev-ref", "HEAD")
GIT_DIRTY = bool(git("status", "--porcelain"))
RUN_DATE = date.today().isoformat()

print(f"colab     : {IN_COLAB}")
print(f"repo root : {REPO_ROOT}")
print(f"git sha   : {GIT_SHA} on {GIT_BRANCH}" + ("   WORKING TREE DIRTY" if GIT_DIRTY else ""))
print(f"run date  : {RUN_DATE}")

colab     : False
repo root : /Users/gina/Documents/AG_PRAXIS
git sha   : 8b78842 on main   WORKING TREE DIRTY
run date  : 2026-08-16


Configuration and inputs. The window arrays live on Drive because they are too large
for the repository, and the manifest that records how they were cut is committed, so the
two are checked against each other rather than either being trusted alone. Nothing is
written, so no output directory is created.

In [5]:
import json

import numpy as np
import pandas as pd

from src import captures as cap
from src import interventions as iv
from src import inventory as inv

CFG = inv.load_config(REPO_ROOT)

SEED = CFG["seed"]
WINDOW = int(CFG["sequence"]["window"])
STRIDE = int(CFG["sequence"]["stride"])
ARTIFACTS = Path(CFG["paths"]["artifacts"])


def first_existing(candidates, what):
    found = next((Path(p) for p in candidates if Path(p).exists()), None)
    if found is None:
        raise FileNotFoundError(f"{what} not found. Looked in: {[str(p) for p in candidates]}")
    return found


MANIFEST_PATH = first_existing(
    [REPO_ROOT / "data" / "processed" / "NB04_manifest.json",
     ARTIFACTS / "NB04" / "NB04_manifest.json"],
    "the manifest recording how the windows were cut",
)
ARRAY_DIR = first_existing([ARTIFACTS / "NB04"], "the saved sequence arrays")

MANIFEST = json.loads(MANIFEST_PATH.read_text())

FEATURES = list(MANIFEST["columns"]["kept"])
CLASSES = sorted(MANIFEST["arrays"]["sequences_train"]["by_class"])
SEQUENCES = {
    part: dict(MANIFEST["arrays"][f"sequences_{part}"]["by_class"])
    for part in ("train", "val", "test")
}

pd.set_option("display.max_rows", 400)
pd.set_option("display.width", 240)

print(f"manifest : {MANIFEST_PATH}")
print(f"arrays   : {ARRAY_DIR}")
print(f"seed     : {SEED}")
print(f"window, stride : {WINDOW}, {STRIDE}")
print(f"features : {len(FEATURES)}")
print(f"classes  : {len(CLASSES)}")
print("nothing is written by this notebook")

FileNotFoundError: the saved sequence arrays not found. Looked in: ['/content/drive/MyDrive/AG_PRAXIS_artifacts/NB04']

Reading the windows. The array file carries more than the windows and their labels: it
also records, for every window, which recording it was cut from. That column is what this
notebook is for, and it is the one thing the training runs so far have not read.

The three checks before anything is returned are the same ones any run makes. The columns
have to be the ones the manifest lists, the classes have to be the ones it lists, and the
file has to have been cut at the window and stride this configuration names. A file that
fails any of them describes a different corpus, and every count below it would be a count
of something else.

In [ ]:
def read_partition(name):
    path = ARRAY_DIR / f"sequences_{name}.npz"
    if not path.exists():
        raise FileNotFoundError(f"{path} is missing. The preprocessing step writes it.")
    with np.load(path, allow_pickle=False) as npz:
        if [str(v) for v in npz["features"]] != FEATURES:
            raise ValueError(f"{path.name} holds different columns than the manifest lists")
        if [str(v) for v in npz["classes"]] != CLASSES:
            raise ValueError(f"{path.name} holds different classes than the manifest lists")
        if (int(npz["window"]), int(npz["stride"])) != (WINDOW, STRIDE):
            raise ValueError(f"{path.name} was cut at a different window or stride")
        X = npz["X"]
        y = npz["y"].astype("int64")
        recording_codes = npz["recording"].astype("int64")
        recordings = [str(v) for v in npz["recordings"]]
    print(f"  {path.name:<24} {str(X.shape):>22}   {X.dtype}")
    assert X.shape[1:] == (WINDOW, len(FEATURES))
    assert len(recording_codes) == len(y) == len(X)
    assert np.isfinite(X).all(), f"{path.name} holds a value that is not finite"
    return {"X": X, "y": y, "recording_codes": recording_codes, "recordings": recordings}


print("reading the sequence arrays")
TRAIN = read_partition("train")
TEST = read_partition("test")

print()
print(f"train windows : {len(TRAIN['y']):,}")
print(f"test windows  : {len(TEST['y']):,}")
print(f"each window   : {WINDOW} records of {len(FEATURES)} features, already scaled")
print(f"recordings named in the training file : {len(TRAIN['recordings'])}")
print(f"recordings named in the test file     : {len(TEST['recordings'])}")

for part, held in (("train", TRAIN), ("test", TEST)):
    counts = {CLASSES[c]: int(n) for c, n in enumerate(np.bincount(held["y"], minlength=len(CLASSES)))}
    assert counts == SEQUENCES[part], (
        f"the {part} array holds different per-class counts than the manifest records"
    )
print()
print("per-class counts in both partitions agree with the manifest")

From recording to session. A recording name carries the partition it belongs to and,
for a class recorded more than once, a chunk number. The session identifier is what is
left when those are stripped, and the project already has one rule for doing that, so it
is used here rather than a second rule being written that could disagree with it.

Every window then carries an integer code for its session, and those codes are what a
later experiment would group by.

In [ ]:
def capture_of(recording_name):
    """The capture a recording belongs to, by the project's own naming rule."""
    return cap.parse_capture(f"{recording_name}.pcap.csv")["capture_id"]


CAPTURE_OF_RECORDING = {name: capture_of(name) for name in TRAIN["recordings"]}
TRAIN_CAPTURES = [CAPTURE_OF_RECORDING[TRAIN["recordings"][c]] for c in TRAIN["recording_codes"]]

GROUP_CODES, GROUP_NAMES = iv.group_codes(TRAIN_CAPTURES)
GROUP_SIZES = iv.group_sizes(GROUP_CODES, len(GROUP_NAMES))
N_GROUPS = len(GROUP_NAMES)

CLASS_OF_GROUP = {}
for code, label_code in zip(GROUP_CODES, TRAIN["y"]):
    CLASS_OF_GROUP.setdefault(GROUP_NAMES[code], CLASSES[label_code])

CAPTURES_PER_CLASS = {}
for group, label in CLASS_OF_GROUP.items():
    CAPTURES_PER_CLASS.setdefault(label, []).append(group)

MULTI = {k: sorted(v) for k, v in CAPTURES_PER_CLASS.items() if len(v) > 1}
SINGLE = {k: sorted(v) for k, v in CAPTURES_PER_CLASS.items() if len(v) == 1}

# A capture belongs to exactly one class, so a capture that reached two would mean the
# naming rule had merged two recordings that are not the same session.
for group in GROUP_NAMES:
    labels = {CLASSES[c] for c, g in zip(TRAIN["y"], GROUP_CODES) if g == GROUP_NAMES.index(group)}
    assert len(labels) == 1, f"capture {group} carries more than one class: {sorted(labels)}"

print(f"captures over the training windows : {N_GROUPS}")
print(f"classes recorded more than once    : {len(MULTI)}")
print(f"classes recorded once              : {len(SINGLE)}")
print()
print("the classes recorded more than once, and their captures")
for label in sorted(MULTI):
    windows = sum(int(GROUP_SIZES[GROUP_NAMES.index(g)]) for g in MULTI[label])
    print(f"  {label:<12} {len(MULTI[label])} captures   {windows:>7,} training windows   "
          f"{', '.join(MULTI[label])}")

What the experiment would have to work with. The windows of the classes recorded more
than once are the only ones where a session can be asked about while the class is held
still, so their count is the size of the ground available, and everything else is
background.

In [ ]:
MULTI_CLASSES = sorted(MULTI)
MULTI_CAPTURES = sorted({g for label in MULTI_CLASSES for g in MULTI[label]})
MULTI_MASK = np.isin(TRAIN["y"], [CLASSES.index(c) for c in MULTI_CLASSES])

N_MULTI_WINDOWS = int(MULTI_MASK.sum())
N_TRAIN_WINDOWS = int(len(TRAIN["y"]))

POOLED_CHANCE = 1.0 / len(MULTI_CAPTURES)
HELD_FIXED_CHANCE = float(np.mean([1.0 / len(MULTI[c]) for c in MULTI_CLASSES]))

SUMMARY = pd.DataFrame([
    {
        "class": label,
        "captures": len(MULTI[label]),
        "train windows": int(GROUP_SIZES[[GROUP_NAMES.index(g) for g in MULTI[label]]].sum()),
        "smallest capture": int(min(GROUP_SIZES[GROUP_NAMES.index(g)] for g in MULTI[label])),
        "largest capture": int(max(GROUP_SIZES[GROUP_NAMES.index(g)] for g in MULTI[label])),
    }
    for label in MULTI_CLASSES
])

print(SUMMARY.to_string(index=False))
print()
print(f"classes recorded more than once : {len(MULTI_CLASSES)}")
print(f"captures across them            : {len(MULTI_CAPTURES)} of {N_GROUPS}")
print(f"training windows in them        : {N_MULTI_WINDOWS:,} of {N_TRAIN_WINDOWS:,}"
      f"   ({100 * N_MULTI_WINDOWS / N_TRAIN_WINDOWS:.1f}%)")
print()
print("guessing rates for a session identified from one of these windows")
print(f"  pooled over all {len(MULTI_CAPTURES)} captures        : {POOLED_CHANCE:.4f}")
print(f"  with the attack class held fixed  : {HELD_FIXED_CHANCE:.4f}")
print()
print("The second is the rate that matters. Pooled over every capture, naming the capture "
      "would name")
print("the class as well, so a model doing it well would only be telling the classes apart "
      "under")
print("another name.")

assert MULTI_MASK.sum() > 0, "no window belongs to a class recorded more than once"
assert len(MULTI_CAPTURES) + len(SINGLE) == N_GROUPS, (
    "the captures of the multi-capture classes and the single-capture classes do not "
    "account for every capture"
)

One last thing to look at before any of this is built on, because it decides where a
session can be measured at all. A session can only be told from another session of the
same class if that class has more than one session in the partition being measured. That
holds on the training partition and it is worth checking whether it holds on the others,
since a number measured only where the training objective already looked is a weaker
number than one measured on held-out data.

In [ ]:
def captures_per_class(held):
    """How many distinct captures each class has in one partition."""
    names = [capture_of(held["recordings"][c]) for c in held["recording_codes"]]
    per_class = {}
    for label_code, group in zip(held["y"], names):
        per_class.setdefault(CLASSES[label_code], set()).add(group)
    return {label: len(groups) for label, groups in sorted(per_class.items())}


TRAIN_PER_CLASS = captures_per_class(TRAIN)
TEST_PER_CLASS = captures_per_class(TEST)

PARTITIONS = pd.DataFrame([
    {
        "class": label,
        "captures in train": TRAIN_PER_CLASS.get(label, 0),
        "captures in test": TEST_PER_CLASS.get(label, 0),
    }
    for label in CLASSES
])

print(PARTITIONS.to_string(index=False))
print()
print(f"classes with more than one capture in train : "
      f"{sum(1 for n in TRAIN_PER_CLASS.values() if n > 1)}")
print(f"classes with more than one capture in test  : "
      f"{sum(1 for n in TEST_PER_CLASS.values() if n > 1)}")
print()
print("Where a class has one capture in a partition, asking which of its captures a window "
      "came")
print("from has one answer, so the question cannot be put on that partition at all.")